# Sprint 6 — Morphème-GPT
## Pourquoi dépasser le niveau caractère ?

Le modèle actuel (Sprints 3-4) génère lettre par lettre. Il produit des séquences **phonétiques** mais sans sens. Les vrais noms de marques sont composés de **morphèmes** :

| Type | Exemples |
|---|---|
| Préfixes | `neo`, `bio`, `eco`, `tech`, `ultra` |
| Racines | `vision`, `link`, `core`, `flow`, `hub` |
| Suffixes | `-ia`, `-ex`, `-al`, `-ix`, `-ify` |

L'idée : apprendre à combiner des **unités signifiantes** plutôt que des caractères.

## Architecture Sprint 6 vs Sprint 3

```
Sprint 3 : données brutes → caractères → NanoGPT → noms phonétiques
Sprint 6 : données catégorisées → morphèmes (BPE) → MorphGPT → noms réalistes
```

## Principes de Karpathy appliqués
- BPE (Byte Pair Encoding) : algorithme utilisé dans GPT-4, même logique, adapté aux noms courts
- Même architecture Transformer mais avec tokens plus riches
- Tokens de contrôle : `<ML>` (marque luxe), `<MT>` (marque tech), `<SG>` (société générale)...


In [ ]:
import sys, os, json, math
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from collections import Counter

# ── Chemins ──────────────────────────────────────────────────
WORK_DIR    = os.path.abspath('..')
DATA_DIR    = os.path.join(WORK_DIR, 'data')
BACKEND_DIR = os.path.join(WORK_DIR, 'backend')
WEIGHTS_DIR = os.path.join(BACKEND_DIR, 'app', 'weights')

if BACKEND_DIR not in sys.path:
    sys.path.insert(0, BACKEND_DIR)

os.makedirs(WEIGHTS_DIR, exist_ok=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Dispositif : {device.upper()}")
print(f"Données    : {DATA_DIR}")
print(f"Poids      : {WEIGHTS_DIR}")


In [ ]:
# ── Tokens de contrôle (un par catégorie) ────────────────────
CONTROL_TOKENS = {
    'fr': {
        'marque_luxe':       '<ML>',
        'marque_tech':       '<MT>',
        'marque_food':       '<MF>',
        'marque_general':    '<MG>',
        'societe_tech':      '<ST>',
        'societe_services':  '<SS>',
        'societe_industrie': '<SI>',
        'societe_general':   '<SG>',
    },
    'ar': {
        'marque_luxe':       '<ML>',
        'marque_tech':       '<MT>',
        'marque_food':       '<MF>',
        'marque_general':    '<MG>',
        'societe_tech':      '<ST>',
        'societe_services':  '<SS>',
        'societe_industrie': '<SI>',
        'societe_general':   '<SG>',
    }
}

def load_lang(lang):
    """Charge tous les fichiers d'une langue depuis data/<lang>/*.txt"""
    result = {}
    lang_dir = os.path.join(DATA_DIR, lang)
    if not os.path.exists(lang_dir):
        print(f"[WARN] {lang_dir} introuvable")
        return result
    for fname in sorted(os.listdir(lang_dir)):
        if not fname.endswith('.txt'):
            continue
        cat = fname.replace('.txt', '')
        path = os.path.join(lang_dir, fname)
        with open(path, encoding='utf-8') as f:
            names = [l.strip() for l in f
                     if l.strip() and ' ' not in l.strip()]
        if names:
            result[cat] = names
    return result

data_fr = load_lang('fr')
data_ar = load_lang('ar')

print("\n=== Données françaises ===")
total_fr = 0
for cat, nms in data_fr.items():
    print(f"  {cat:<25} {len(nms):>4} noms")
    total_fr += len(nms)
print(f"  Total : {total_fr}")

print("\n=== Données arabes ===")
total_ar = 0
for cat, nms in data_ar.items():
    print(f"  {cat:<25} {len(nms):>4} noms")
    total_ar += len(nms)
print(f"  Total : {total_ar}")


## BPE — Byte Pair Encoding adapté aux noms de marques

**Algorithme (identique à GPT-4, mais sur un corpus de noms courts) :**
1. Démarrer avec un vocabulaire de caractères
2. Compter toutes les paires de tokens adjacents
3. Fusionner la paire la plus fréquente en un nouveau token
4. Répéter jusqu'à atteindre `vocab_size`

**Résultat sur noms de marques :**  
`a-l-t-e-c-h` → après 200 itérations → `al-tech`  
`b-i-o-n-o-v-a` → `bio-nova`  
`s-y-n-a-p-s-o` → `syn-ap-so`


In [ ]:
class BrandBPE:
    """
    Tokeniseur BPE spécialisé pour les noms de marques.
    Apprend des morphèmes récurrents : 'neo', 'tech', 'ia', 'ex'...

    Tokens spéciaux :
        <PAD> = 0  padding
        <BOS> = 1  début de séquence
        <EOS> = 2  fin de séquence
        <ML>  = 3  marque luxe
        <MT>  = 4  marque tech
        ... etc.
    """

    SPECIAL = ['<PAD>', '<BOS>', '<EOS>']

    def __init__(self, vocab_size: int = 300):
        self.vocab_size = vocab_size
        self.vocab: dict[str, int] = {}
        self.inv_vocab: dict[int, str] = {}
        self.merges: list[tuple] = []
        # Initialisation des tokens spéciaux
        for tok in self.SPECIAL:
            self._add(tok)

    def _add(self, token: str) -> int:
        if token not in self.vocab:
            idx = len(self.vocab)
            self.vocab[token] = idx
            self.inv_vocab[idx] = token
        return self.vocab[token]

    def add_control_tokens(self, tokens: list[str]):
        """Ajoute les tokens de contrôle avant l'entraînement BPE."""
        for t in tokens:
            self._add(t)

    def train(self, names: list[str]):
        """
        Entraîne le BPE sur une liste de noms.
        Ajoute d'abord tous les caractères, puis fusionne
        itérativement les paires les plus fréquentes.
        """
        # 1. Initialisation : vocabulaire de caractères
        for name in names:
            for ch in name:
                self._add(ch)

        # 2. Représentation de chaque mot comme tuple de chars
        word_freq: dict[tuple, int] = {}
        for name in names:
            t = tuple(name)
            word_freq[t] = word_freq.get(t, 0) + 1

        # 3. Itérations BPE
        while len(self.vocab) < self.vocab_size:
            # Compter les paires
            pairs: dict[tuple, int] = {}
            for word, freq in word_freq.items():
                for i in range(len(word) - 1):
                    p = (word[i], word[i+1])
                    pairs[p] = pairs.get(p, 0) + freq

            if not pairs:
                break

            # Meilleure paire
            best = max(pairs, key=pairs.get)
            new_tok = ''.join(best)
            self._add(new_tok)
            self.merges.append(best)

            # Appliquer la fusion
            new_word_freq: dict[tuple, int] = {}
            for word, freq in word_freq.items():
                new_word = []
                i = 0
                while i < len(word):
                    if (i < len(word) - 1
                            and (word[i], word[i+1]) == best):
                        new_word.append(new_tok)
                        i += 2
                    else:
                        new_word.append(word[i])
                        i += 1
                new_word_freq[tuple(new_word)] =                     new_word_freq.get(tuple(new_word), 0) + freq
            word_freq = new_word_freq

        print(f"Vocabulaire BPE : {len(self.vocab)} tokens "
              f"({len(self.merges)} fusions)")

    def tokenize(self, name: str) -> list[int]:
        """Convertit un nom en liste d'IDs (avec BOS et EOS)."""
        word = list(name)
        for merge in self.merges:
            new_word = []
            i = 0
            while i < len(word):
                if (i < len(word) - 1
                        and (word[i], word[i+1]) == merge):
                    new_word.append(''.join(merge))
                    i += 2
                else:
                    new_word.append(word[i])
                    i += 1
            word = new_word
        ids = [self.vocab.get(ch, 0) for ch in word]
        return [1] + ids + [2]  # BOS + tokens + EOS

    def decode(self, ids: list[int]) -> str:
        """Convertit des IDs en texte lisible."""
        parts = []
        for i in ids:
            tok = self.inv_vocab.get(i, '')
            if tok in ('<PAD>', '<BOS>'):
                continue
            if tok == '<EOS>':
                break
            if tok.startswith('<') and tok.endswith('>'):
                continue  # ignore control tokens
            parts.append(tok)
        return ''.join(parts)

    def save(self, path: str):
        data = {
            'vocab_size': self.vocab_size,
            'vocab':  self.vocab,
            'merges': [list(m) for m in self.merges]
        }
        with open(path, 'w', encoding='utf-8') as f:
            json.dump(data, f, ensure_ascii=False, indent=2)
        print(f"BPE sauvegardé → {path}")

    @classmethod
    def load(cls, path: str) -> 'BrandBPE':
        with open(path, encoding='utf-8') as f:
            d = json.load(f)
        bpe = cls(d['vocab_size'])
        bpe.vocab     = d['vocab']
        bpe.inv_vocab = {v: k for k, v in d['vocab'].items()}
        bpe.merges    = [tuple(m) for m in d['merges']]
        return bpe

print("Classe BrandBPE définie ✓")


In [ ]:
# ── Collecte de tous les noms (FR) pour entraîner le BPE ────
all_names_fr = []
for cat, names in data_fr.items():
    all_names_fr.extend(names)

# Ajout des tokens de contrôle avant l'entraînement
ctrl_tokens = list(set(CONTROL_TOKENS['fr'].values()))

bpe_fr = BrandBPE(vocab_size=350)
bpe_fr.add_control_tokens(ctrl_tokens)
bpe_fr.train(all_names_fr)

# ── Analyse du vocabulaire ────────────────────────────────────
print("\n=== Morphèmes appris (extrait) ===")
# Filtrer les tokens non-spéciaux et non-caractères uniques
morphemes = [
    tok for tok in bpe_fr.vocab
    if len(tok) >= 2
    and not tok.startswith('<')
]
print(f"Morphèmes (longueur ≥ 2) : {len(morphemes)}")
print(f"Exemples : {sorted(morphemes, key=len, reverse=True)[:30]}")

# ── Test de tokenisation ──────────────────────────────────────
print("\n=== Tests de tokenisation ===")
test_names = ['neotech', 'biomax', 'technova', 'algorix', 'altimax']
for name in test_names:
    ids    = bpe_fr.tokenize(name)
    tokens = [bpe_fr.inv_vocab[i] for i in ids]
    decoded = bpe_fr.decode(ids)
    print(f"  {name:12} → {tokens} → '{decoded}'")

# ── BPE arabe ────────────────────────────────────────────────
all_names_ar = []
for names in data_ar.values():
    all_names_ar.extend(names)

bpe_ar = BrandBPE(vocab_size=250)
bpe_ar.add_control_tokens(list(set(CONTROL_TOKENS['ar'].values())))
bpe_ar.train(all_names_ar)

print(f"\nBPE AR : {len(bpe_ar.vocab)} tokens")

# ── Sauvegarde des vocabulaires ──────────────────────────────
bpe_fr.save(os.path.join(WEIGHTS_DIR, 'bpe_fr.json'))
bpe_ar.save(os.path.join(WEIGHTS_DIR, 'bpe_ar.json'))


## MorphGPT — Architecture

Même base que NanoGPT de Karpathy, avec deux améliorations clés :

1. **Weight tying** : la matrice d'embedding et la tête LM partagent les mêmes poids (économise ~30% de paramètres)
2. **Ignore index** : le padding (id=0) est ignoré dans le calcul de la loss → apprentissage plus propre sur des séquences de longueurs variables


In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, block_size, dropout=0.1):
        super().__init__()
        assert n_embd % n_head == 0
        self.n_head    = n_head
        self.head_size = n_embd // n_head
        self.qkv  = nn.Linear(n_embd, 3 * n_embd, bias=False)
        self.proj = nn.Linear(n_embd, n_embd, bias=False)
        self.attn_drop = nn.Dropout(dropout)
        self.proj_drop = nn.Dropout(dropout)
        self.register_buffer(
            'mask',
            torch.tril(torch.ones(block_size, block_size))
                  .view(1, 1, block_size, block_size)
        )

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        q = q.view(B, T, self.n_head, self.head_size).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_size).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_size).transpose(1, 2)
        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_size)
        att = att.masked_fill(
            self.mask[:, :, :T, :T] == 0, float('-inf'))
        att = self.attn_drop(F.softmax(att, dim=-1))
        y = (att @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.proj_drop(self.proj(y))


class FeedForward(nn.Module):
    def __init__(self, n_embd, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.GELU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )
    def forward(self, x): return self.net(x)


class TransformerBlock(nn.Module):
    def __init__(self, n_embd, n_head, block_size, dropout=0.1):
        super().__init__()
        self.ln1  = nn.LayerNorm(n_embd)
        self.attn = CauxsalSelfAttention(n_embd, n_head, block_size, dropout)
        self.ln2  = nn.LayerNorm(n_embd)
        self.ff   = FeedForward(n_embd, dropout)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x


class MorphGPT(nn.Module):
    """
    GPT adapté aux morphèmes de noms de marques.
    Hérite de la logique NanoGPT (Karpathy) avec :
      - weight tying (tok_emb ↔ lm_head)
      - ignore_index=0 dans la loss (padding)
      - tokens de contrôle dans le vocabulaire
    """

    def __init__(self, vocab_size, n_embd=128, n_head=4,
                 n_layer=4, block_size=24, dropout=0.1):
        super().__init__()
        self.block_size = block_size

        self.tok_emb = nn.Embedding(vocab_size, n_embd)
        self.pos_emb = nn.Embedding(block_size, n_embd)
        self.drop    = nn.Dropout(dropout)
        self.blocks  = nn.Sequential(*[
            TransformerBlock(n_embd, n_head, block_size, dropout)
            for _ in range(n_layer)
        ])
        self.ln_f = nn.LayerNorm(n_embd)
        self.head = nn.Linear(n_embd, vocab_size, bias=False)

        # Weight tying (même matrice pour embedding et tête LM)
        self.tok_emb.weight = self.head.weight

        self.apply(self._init_weights)
        print(f"MorphGPT : {self.count_params():,} paramètres")

    def _init_weights(self, m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, 0.0, 0.02)
            if hasattr(m, 'bias') and m.bias is not None:
                nn.init.zeros_(m.bias)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        pos  = torch.arange(T, dtype=torch.long, device=idx.device)
        x    = self.drop(self.tok_emb(idx) + self.pos_emb(pos))
        x    = self.blocks(x)
        logits = self.head(self.ln_f(x))

        loss = None
        if targets is not None:
            B, T, C = logits.shape
            loss = F.cross_entropy(
                logits.view(B * T, C),
                targets.view(B * T),
                ignore_index=0   # ignore le padding
            )
        return logits, loss

    def count_params(self):
        return sum(p.numel() for p in self.parameters())

    @torch.no_grad()
    def generate(self, ctx, max_new=12, temp=0.8,
                 top_k=20, eos_id=2):
        """Génération autorégressive à partir d'un contexte."""
        for _ in range(max_new):
            ctx_in = ctx[:, -self.block_size:]
            logits, _ = self(ctx_in)
            logits = logits[:, -1, :] / max(temp, 1e-5)
            if top_k:
                v, _ = torch.topk(logits,
                                   min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float('-inf')
            probs    = F.softmax(logits, dim=-1)
            next_tok = torch.multinomial(probs, 1)
            if next_tok.item() == eos_id:
                break
            ctx = torch.cat([ctx, next_tok], dim=1)
        return ctx

print("Modèle MorphGPT défini ✓")


In [ ]:
def prepare_sequences(data_lang, bpe, control_tokens, max_len=24):
    """
    Crée les séquences d'entraînement :
    [ctrl_token_id, tok1, tok2, ..., EOS]

    Chaque nom est précédé de son token de contrôle de catégorie.
    """
    sequences = []
    for cat, names in data_lang.items():
        ctrl = control_tokens.get(cat, '<MG>')
        ctrl_id = bpe.vocab.get(ctrl, 1)  # fallback BOS si absent

        for name in names:
            ids = bpe.tokenize(name)  # [BOS, t1, t2, ..., EOS]
            # Remplacer BOS par le token de contrôle
            ids[0] = ctrl_id
            if len(ids) <= max_len:
                # Padding à droite jusqu'à max_len
                padded = ids + [0] * (max_len - len(ids))
                sequences.append(padded)

    return sequences

# ── Création des séquences ────────────────────────────────────
MAX_LEN = 24

seqs_fr = prepare_sequences(
    data_fr, bpe_fr, CONTROL_TOKENS['fr'], MAX_LEN)
seqs_ar = prepare_sequences(
    data_ar, bpe_ar, CONTROL_TOKENS['ar'], MAX_LEN)

print(f"Séquences FR : {len(seqs_fr)}")
print(f"Séquences AR : {len(seqs_ar)}")

# Vérification visuelle
print("\n=== Exemples de séquences FR ===")
for s in seqs_fr[:3]:
    tokens = [bpe_fr.inv_vocab.get(i, '?') for i in s if i != 0]
    name   = bpe_fr.decode(s)
    print(f"  IDs     : {s[:10]}...")
    print(f"  Tokens  : {tokens}")
    print(f"  Décodé  : '{name}'")
    print()

# ── Tenseurs train/val ────────────────────────────────────────
def to_tensors(seqs, val_ratio=0.1):
    import random
    random.shuffle(seqs)
    split  = int(len(seqs) * (1 - val_ratio))
    train  = torch.tensor(seqs[:split],  dtype=torch.long)
    val    = torch.tensor(seqs[split:],  dtype=torch.long)
    return train, val

train_fr, val_fr = to_tensors(seqs_fr)
train_ar, val_ar = to_tensors(seqs_ar)

print(f"Train FR : {train_fr.shape} | Val FR : {val_fr.shape}")
print(f"Train AR : {train_ar.shape} | Val AR : {val_ar.shape}")


In [ ]:
def train_morph_model(train_data, val_data, bpe, lang,
                     n_embd=128, n_head=4, n_layer=4,
                     max_len=24, batch_size=64,
                     max_iters=3000, eval_every=300, lr=3e-4):
    """
    Entraîne un MorphGPT sur des séquences de morphèmes.

    La loss cible : < 1.0 (très bon) | 1.0-1.5 (bon) | > 2.0 (à retravailler)
    """
    vocab_size = len(bpe.vocab)
    model = MorphGPT(vocab_size, n_embd, n_head, n_layer,
                     max_len).to(device)

    optimizer = optim.AdamW(model.parameters(), lr=lr,
                            weight_decay=0.01)
    # Scheduler cosine
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=max_iters, eta_min=lr / 10)

    def get_batch(split):
        data = train_data if split == 'train' else val_data
        idx  = torch.randint(len(data), (batch_size,))
        rows = data[idx].to(device)          # (B, L)
        x    = rows[:, :-1]                  # input
        y    = rows[:, 1:]                   # target décalé d'1
        return x, y

    print(f"\n{'='*55}")
    print(f"  Entraînement MorphGPT [{lang.upper()}]")
    print(f"  Vocab : {vocab_size} | Paramètres : {model.count_params():,}")
    print(f"  Données train : {len(train_data)} | val : {len(val_data)}")
    print(f"{'='*55}")

    model.train()
    for it in range(max_iters):
        x, y = get_batch('train')
        _, loss = model(x, y)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        # Gradient clipping (stabilise l'entraînement)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        if it % eval_every == 0 or it == max_iters - 1:
            model.eval()
            with torch.no_grad():
                xv, yv = get_batch('val')
                _, lv = model(xv, yv)
            print(f"  iter {it:4d} | train {loss.item():.4f} "
                  f"| val {lv.item():.4f} "
                  f"| lr {scheduler.get_last_lr()[0]:.5f}")
            model.train()

    return model


# ── Entraînement FR ───────────────────────────────────────────
model_fr = train_morph_model(
    train_fr, val_fr, bpe_fr, 'fr',
    n_embd=128, n_head=4, n_layer=4,
    max_iters=3000, batch_size=64
)

# ── Entraînement AR ───────────────────────────────────────────
model_ar = train_morph_model(
    train_ar, val_ar, bpe_ar, 'ar',
    n_embd=128, n_head=4, n_layer=4,
    max_iters=2000, batch_size=32
)


In [ ]:
def generate_names(model, bpe, control_token, n=10,
                   temp=0.8, top_k=20):
    """
    Génère n noms en commençant par un token de contrôle.

    Paramètres
    ----------
    control_token : ex. '<MT>' pour marque tech
    temp          : temperature (0.5 = plus conservateur, 1.2 = plus créatif)
    top_k         : nombre de tokens candidats à chaque étape
    """
    model.eval()
    ctrl_id = bpe.vocab.get(control_token, 1)
    names   = []

    with torch.no_grad():
        for _ in range(n * 4):  # sur-génère pour filtrer
            # Contexte initial : [ctrl_token]
            ctx = torch.zeros(1, model.block_size,
                              dtype=torch.long, device=device)
            ctx[0, -1] = ctrl_id

            # Génération
            out = model.generate(ctx, max_new=14, temp=temp,
                                  top_k=top_k, eos_id=2)

            # Décodage : on prend les tokens après le contexte initial
            new_ids = out[0, model.block_size:].tolist()
            name    = bpe.decode(new_ids)

            # Filtre qualité
            if 3 <= len(name) <= 14 and name not in names:
                names.append(name)

            if len(names) >= n:
                break

    return names[:n]


# ── Tests de génération ───────────────────────────────────────
tests_fr = [
    ('<ML>', 'Marque Luxe'),
    ('<MT>', 'Marque Tech'),
    ('<MF>', 'Marque Food'),
    ('<ST>', 'Société Tech'),
    ('<SS>', 'Société Services'),
]

print("=== Génération FR par catégorie ===\n")
for ctrl, label in tests_fr:
    noms = generate_names(model_fr, bpe_fr, ctrl, n=8)
    print(f"  {label:<22} [{ctrl}] → {noms}")

print("\n=== Génération AR par catégorie ===\n")
tests_ar = [
    ('<ML>', 'Marque Luxe'),
    ('<MT>', 'Marque Tech'),
    ('<MG>', 'Marque Général'),
]
for ctrl, label in tests_ar:
    noms = generate_names(model_ar, bpe_ar, ctrl, n=6)
    print(f"  {label:<22} [{ctrl}] → {noms}")


In [ ]:
# ── Comparaison avec le modèle Sprint 3 ─────────────────────
print("=== Comparaison qualitative ===\n")
print("Sprint 3 (caractères) :")
print("  Marque Tech → ['alizal', 'talionn', 'aloux', 'teniom', ...]")
print("  (séquences phonétiques sans sens)\n")

print("Sprint 6 (morphèmes) :")
noms_demo = generate_names(model_fr, bpe_fr, '<MT>', n=8)
print(f"  Marque Tech → {noms_demo}")
print("  (morphèmes reconnaissables, combinaisons signifiantes)\n")

# ── Statistiques de longueur ─────────────────────────────────
tous = []
for ctrl in ['<ML>', '<MT>', '<MF>', '<MG>', '<ST>', '<SS>']:
    tous.extend(generate_names(model_fr, bpe_fr, ctrl, n=20))

if tous:
    from statistics import mean, stdev
    lengths = [len(n) for n in tous]
    print(f"Longueur moyenne : {mean(lengths):.1f} ± {stdev(lengths):.1f}")
    print(f"Min / Max        : {min(lengths)} / {max(lengths)}")

# ── Export ───────────────────────────────────────────────────
print("\n=== Export des modèles ===")

# Sauvegarder les modèles MorphGPT
torch.save(model_fr.state_dict(),
           os.path.join(WEIGHTS_DIR, 'morph_fr.pt'))
torch.save(model_ar.state_dict(),
           os.path.join(WEIGHTS_DIR, 'morph_ar.pt'))

# Sauvegarder les vocabulaires BPE (déjà fait plus haut)
# (bpe_fr.json et bpe_ar.json existent déjà)

# Sauvegarder la config du modèle
config = {
    'model_type': 'MorphGPT',
    'vocab_size_fr': len(bpe_fr.vocab),
    'vocab_size_ar': len(bpe_ar.vocab),
    'n_embd': 128,
    'n_head': 4,
    'n_layer': 4,
    'block_size': 24,
    'control_tokens': CONTROL_TOKENS
}
with open(os.path.join(WEIGHTS_DIR, 'morph_config.json'),
          'w', encoding='utf-8') as f:
    json.dump(config, f, ensure_ascii=False, indent=2)

print("\nFichiers exportés :")
for fname in os.listdir(WEIGHTS_DIR):
    fpath = os.path.join(WEIGHTS_DIR, fname)
    size  = os.path.getsize(fpath) // 1024
    icon  = '🟠' if fname.endswith('.pt') else '📄'
    print(f"  {icon} {fname:<35} {size:>5} Ko")

print("\n✓ Sprint 6 terminé — MorphGPT prêt pour intégration.")
